# Adult Census Income - ETL Pipeline
## Phase: Data Cleaning & Transformation

### Step 1: Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np

# Load data, skipinitialspace helps remove leading spaces in strings like ' ?'
df = pd.read_csv('../data/raw/adult.csv', skipinitialspace=True)
print(f"Original Shape: {df.shape}")
display(df.head())

### Step 2: Data Cleaning
Complete cleaning with simple, readable steps.

In [ ]:
# 1. Clean column names (snake_case)
df.columns = df.columns.str.lower().str.replace('-', '_')

# 2. Drop the 'fnlwgt' column
if 'fnlwgt' in df.columns:
    df.drop('fnlwgt', axis=1, inplace=True)

# 3. Strip extra spaces from string columns
object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    df[col] = df[col].str.strip()

# 4. Handle Missing Values: Replace '?' with NaN, then fill with Mode
df.replace('?', np.nan, inplace=True)

for col in df.columns:
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)

# 5. Remove Duplicates
df.drop_duplicates(inplace=True)

print("Data Cleaning Complete!")
print(f"Cleaned Shape: {df.shape}")
print(f"Total NaN Values strictly remaining: {df.isnull().sum().sum()}")

### Step 3: Data Transformation

In [ ]:
# 1. Income Binary Encoding (>50K -> 1, <=50K -> 0)
df['income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)

# 2. Age Bucketizing
df['age_group'] = pd.cut(df['age'], bins=[16, 25, 45, 65, 100], labels=['Youth', 'YoungAdult', 'Adult', 'Senior'])

display(df[['age', 'age_group', 'income']].head())

### Step 4: Export to Processed Data

In [ ]:
import os

# Save the perfectly clean data without NaNs
os.makedirs('../data/processed', exist_ok=True)
processed_path = '../data/processed/adult_cleaned.csv'
df.to_csv(processed_path, index=False)

print(f"Data successfully saved to {processed_path}")

# Final validation sample
display(df.sample(5))